# Week 4.2: 51 × 51 grid validation

This companion upgrades the numerical reference and retrains the Stokes-to-Navier–Stokes correction on 51 × 51 nodes. The original 25 × 25 experiment remains a baseline. The same previously inspected case manifest is used, so these are regression cases, not a new blind test.

## Mean velocity relative L2 error (%)

| Train | Test | 25 x 25 | 51 x 51 |
|---|---|---:|---:|
| constant | constant | 0.107% | 0.097% |
| constant | diverse | 19.920% | 16.555% |
| constant | ood_re | 36.982% | 30.700% |
| constant | ood_shape | 37.192% | 33.068% |
| diverse | constant | 2.044% | 1.960% |
| diverse | diverse | 0.917% | 0.810% |
| diverse | ood_re | 10.204% | 6.425% |
| diverse | ood_shape | 8.428% | 7.878% |

Errors at 25 and 51 nodes are measured against their respective same-grid Navier–Stokes solutions. A lower surrogate error does not by itself establish mesh convergence.

## Change in the CFD reference under refinement

The 51-node CFD velocity is linearly sampled at the 25-node locations and compared with the original 25-node CFD velocity. Mean relative differences (%):

| Family | Mean difference |
|---|---:|
| constant | 20.59% |
| diverse | 17.81% |
| ood_re | 32.27% |
| ood_shape | 12.30% |

This is one refinement step, not an asymptotic convergence study.

## Retained figures

![25 versus 51 grid error comparison](../../figures/Cavity_grid25_grid51_errors.png)

![51-node constant-lid speed, streamlines and vorticity](../../figures/Cavity_constant_grid51_streamlines_vorticity.png)

![51-node diverse-lid speed, streamlines and vorticity](../../figures/Cavity_diverse_grid51_streamlines_vorticity.png)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd()
R = ROOT / 'results/stokes_grid51'
display(pd.read_csv(R/'grid_comparison.csv'))
display(pd.read_csv(R/'grid_shift.csv').groupby('family').velocity_grid_shift_rel_l2.mean())
display(pd.read_csv(R/'vortex_metrics.csv'))

In [ ]:
for name in ('Cavity_grid25_grid51_errors.png',
             'Cavity_constant_grid51_streamlines_vorticity.png',
             'Cavity_diverse_grid51_streamlines_vorticity.png'):
    display(Image(filename=str(ROOT/'figures'/name)))

## Reproduce

From the repository root, run `python qa/run_week04_2_grid51.py generate` and then `python qa/run_week04_2_grid51.py train`. The 51-node fields are solved afresh; the network uses the 25-node study's fixed selected architecture (POD rank 24, two tanh layers of width 96, seeds 7/17/27). Training and validation use separate complete cases. The plotted corner extrema are grid-resolved candidates and need finer-grid or independent CFD confirmation.